print("hi")

In [ ]:
!pip install -q roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("ahmads-workspace-wbv9p").project("helmet-detection-tool")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
!pip install -q -U ultralytics
import torch
print("GPU Available:", torch.cuda.is_available())
!nvidia-smi

In [ ]:
!ls /content/drive/MyDrive

In [ ]:
!ls /content/drive/MyDrive/cv_project


In [ ]:
!find /content -maxdepth 4 -name "data.yaml" -not -path "*/dist-packages/*"

In [ ]:
!ls /content/drive/MyDrive/cv_project/roboflow_export

In [ ]:
import os, shutil

BASE = "/content/drive/MyDrive/cv_project"
DATASET = f"{BASE}/dataset"
SRC = f"{BASE}/roboflow_export"

for src_split, dst_split in [("train", "train"), ("valid", "val"), ("test", "test")]:
    for kind in ["images", "labels"]:
        s = f"{SRC}/{src_split}/{kind}"
        d = f"{DATASET}/{kind}/{dst_split}"
        if os.path.isdir(s):
            os.makedirs(d, exist_ok=True)
            for f in os.listdir(s):
                shutil.copy2(os.path.join(s, f), d)
print("Done")

In [ ]:
import yaml

DATA_YAML = f"{DATASET}/data.yaml"
src_yaml = f"{SRC}/data.yaml"

if os.path.exists(src_yaml):
    names = yaml.safe_load(open(src_yaml))["names"]
    names = [names[i] for i in sorted(names)] if isinstance(names, dict) else names
else:
    names = ["helmet", "no-helmet", "person"]

cfg = {
    "path": DATASET,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: n for i, n in enumerate(names)},
}
yaml.safe_dump(cfg, open(DATA_YAML, "w"), sort_keys=False)
print(open(DATA_YAML).read())

In [ ]:
for split in ["train", "val", "test"]:
    n_img = len(os.listdir(f"{DATASET}/images/{split}"))
    n_lbl = len(os.listdir(f"{DATASET}/labels/{split}"))
    print(f"{split:5s}  images: {n_img:4d}   labels: {n_lbl:4d}")

In [ ]:
import os, time, torch
from ultralytics import YOLO

BASE = "/content/drive/MyDrive/cv_project"
DATASET = f"{BASE}/dataset"
DATA_YAML = f"{DATASET}/data.yaml"
RUN_NAME = "yolov8n_baseline"

model = YOLO("yolov8n.pt")

t0 = time.time()
results = model.train(
    data=DATA_YAML,
    epochs=50,          # the task minimum is 30
    imgsz=640,
    batch=16,
    pretrained=True,
    project=f"{BASE}/outputs/training_results",
    name=RUN_NAME,
    exist_ok=True,
)
minutes = (time.time() - t0) / 60
print(f"Training time: {minutes:.1f} min")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os, glob
DATASET = "/content/drive/MyDrive/cv_project/dataset"
for split in ["train", "val", "test"]:
    files = glob.glob(f"{DATASET}/labels/{split}/*.txt")
    poly_files = poly_lines = box_lines = 0
    for f in files:
        has_poly = False
        for line in open(f):
            n = len(line.split())
            if n == 0: continue
            if n > 5: poly_lines += 1; has_poly = True
            else: box_lines += 1
        poly_files += has_poly
    print(f"{split:5s} files: {len(files):3d} | files with polygons: {poly_files:3d} | polygon lines: {poly_lines:4d} | box lines: {box_lines:4d}")

In [ ]:
import os, shutil
from ultralytics import YOLO

BASE = "/content/drive/MyDrive/cv_project"
DATASET = f"{BASE}/dataset"
DATA_YAML = f"{DATASET}/data.yaml"
RUN_DIR = f"{BASE}/outputs/training_results/yolov8n_baseline"

best = f"{RUN_DIR}/weights/best.pt"
model = YOLO(best)

for split in ["val", "test"]:
    m = model.val(data=DATA_YAML, split=split, imgsz=640, plots=True,
                  project=f"{BASE}/outputs/training_results", name=f"eval_{split}", exist_ok=True)
    print(f"\n=== {split.upper()} ===")
    print(f"Precision    : {m.box.mp:.3f}")
    print(f"Recall       : {m.box.mr:.3f}")
    print(f"mAP@0.5      : {m.box.map50:.3f}")
    print(f"mAP@0.5:0.95 : {m.box.map:.3f}")
    print("Per class (P, R, mAP50, mAP50-95):")
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        print(f"  {m.names[c]:10s} {p:.3f} {r:.3f} {ap50:.3f} {ap:.3f}")

In [ ]:
import time, torch
from ultralytics import YOLO

BASE = "/content/drive/MyDrive/cv_project"
DATA_YAML = f"{BASE}/dataset/data.yaml"
RUN_NAME = "yolov8n_cleaned_labels"

model = YOLO("yolov8n.pt")
t0 = time.time()
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    pretrained=True,
    project=f"{BASE}/outputs/training_results",
    name=RUN_NAME,
    exist_ok=True,
)
print(f"Training time: {(time.time() - t0) / 60:.1f} min")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
RUN_DIR = f"{BASE}/outputs/training_results/{RUN_NAME}"
best = f"{RUN_DIR}/weights/best.pt"
model = YOLO(best)

for split in ["val", "test"]:
    m = model.val(data=DATA_YAML, split=split, imgsz=640, plots=True,
                  project=f"{BASE}/outputs/training_results", name=f"eval_{split}_cleaned", exist_ok=True)
    print(f"\n=== {split.upper()} ===")
    print(f"Precision    : {m.box.mp:.3f}")
    print(f"Recall       : {m.box.mr:.3f}")
    print(f"mAP@0.5      : {m.box.map50:.3f}")
    print(f"mAP@0.5:0.95 : {m.box.map:.3f}")
    print("Per class (P, R, mAP50, mAP50-95):")
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        print(f"  {m.names[c]:10s} {p:.3f} {r:.3f} {ap50:.3f} {ap:.3f}")

In [ ]:
import os
from IPython.display import Image, display

BASE = "/content/drive/MyDrive/cv_project"
RUN_DIR = f"{BASE}/outputs/training_results/yolov8n_cleaned_labels"
for p in [f"{RUN_DIR}/confusion_matrix.png",
          f"{RUN_DIR}/results.png",
          f"{BASE}/outputs/training_results/eval_test_cleaned/confusion_matrix.png"]:
    if os.path.exists(p):
        print(p.split("training_results/")[1]); display(Image(p))

In [ ]:
import shutil
from ultralytics import YOLO

best = f"{RUN_DIR}/weights/best.pt"
os.makedirs(f"{BASE}/models", exist_ok=True)
cur = f"{BASE}/models/best.pt"
if os.path.exists(cur) and not os.path.exists(f"{BASE}/models/best_baseline.pt"):
    shutil.copy2(cur, f"{BASE}/models/best_baseline.pt")
shutil.copy2(best, cur)

onnx_path = YOLO(best).export(format="onnx", imgsz=640)
shutil.copy2(onnx_path, f"{BASE}/models/best.onnx")
print("Saved to", f"{BASE}/models")

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

DATASET = f"{BASE}/dataset"
model = YOLO(f"{BASE}/models/best.pt")
preds = model.predict(source=f"{DATASET}/images/test", conf=0.35, save=True,
                      project=f"{BASE}/outputs/predictions", name="test_predictions", exist_ok=True)

files = sorted(glob.glob(f"{preds[0].save_dir}/*.jpg"))[:15]
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for ax, f in zip(axes.ravel(), files):
    ax.imshow(mpimg.imread(f)); ax.set_title(os.path.basename(f)[:20], fontsize=8); ax.axis("off")
for ax in axes.ravel()[len(files):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/cv_project"
DATASET = f"{BASE}/dataset"
names = model.names


def load_gt(path, w, h):
    """Read a YOLO label file and return [(class_id, [x1, y1, x2, y2]), ...] in pixels.
    Handles box lines (5 values) and polygon lines (more than 5 values)."""
    out = []
    if not os.path.exists(path):
        return out
    for line in open(path):
        p = line.split()
        if len(p) < 5:
            continue
        c = int(p[0])
        vals = list(map(float, p[1:]))
        if len(vals) == 4:                      # box: xc, yc, bw, bh
            xc, yc, bw, bh = vals
            x1, y1, x2, y2 = xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2
        else:                                   # polygon: x1 y1 x2 y2 ...
            xs, ys = vals[0::2], vals[1::2]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        out.append((c, [x1 * w, y1 * h, x2 * w, y2 * h]))
    return out


def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0


rows = []
for r in preds:
    img = os.path.basename(r.path)
    stem = os.path.splitext(img)[0]
    h, w = r.orig_shape
    gt = load_gt(f"{DATASET}/labels/test/{stem}.txt", w, h)
    pr = [(int(b.cls), float(b.conf), b.xyxy[0].tolist()) for b in r.boxes]
    used = set()

    # match each prediction (most confident first) to the best unused ground-truth box
    for pc, conf, pb in sorted(pr, key=lambda x: -x[1]):
        best, bj = 0, None
        for j, (gc, gb) in enumerate(gt):
            if j in used:
                continue
            v = iou(pb, gb)
            if v > best:
                best, bj = v, j
        if best >= 0.5:
            used.add(bj)
            gc = gt[bj][0]
            rows.append([img, names[gc], names[pc], round(conf, 3), "correct" if gc == pc else "wrong class"])
        else:
            rows.append([img, "(nothing there)", names[pc], round(conf, 3), "false positive"])

    # ground-truth objects that no prediction matched
    for j, (gc, gb) in enumerate(gt):
        if j not in used:
            rows.append([img, names[gc], "(missed)", None, "false negative"])

err = pd.DataFrame(rows, columns=["image", "actual", "predicted", "confidence", "error_type"])
err["image"] = err["image"].str.replace(r"-612x612.*", "", regex=True).str[:28]   # shorter names

print(err["error_type"].value_counts())
print("\nPer image:")
print(err.groupby("image")["error_type"].value_counts().unstack(fill_value=0))

errors = err[err["error_type"] != "correct"].reset_index(drop=True)
errors.to_csv(f"{BASE}/outputs/predictions/error_analysis_raw.csv", index=False)

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
errors

In [ ]:
import cv2
import matplotlib.pyplot as plt

check = ["gettyimages-1126025369", "gettyimages-2162233192", "gettyimages-2183897891",
         "gettyimages-2220083803", "gettyimages-585617608", "gettyimages-960954612",
         "premium_photo-1783665934212"]

by_name = {os.path.basename(r.path): r for r in preds}
items = []
for name in check:
    key = next((k for k in by_name if k.startswith(name)), None)
    if key:
        items.append((key, by_name[key]))

cols = 4
n_rows = (len(items) + cols - 1) // cols
fig, axes = plt.subplots(n_rows, cols, figsize=(5 * cols, 5 * n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, (key, r) in zip(axes, items):
    img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    stem = os.path.splitext(key)[0]
    for c, (x1, y1, x2, y2) in load_gt(f"{DATASET}/labels/test/{stem}.txt", w, h):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), 2)
        cv2.putText(img, names[c], (int(x1), int(y2) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
    for b in r.boxes:
        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img, f"{names[int(b.cls)]} {float(b.conf):.2f}", (x1, y1 + 14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    ax.imshow(img)
    ax.set_title(key[:26], fontsize=9)
    ax.axis("off")
for ax in axes[len(items):]:
    ax.axis("off")

plt.suptitle("Green = ground truth label, Red = model prediction")
plt.tight_layout()
plt.savefig(f"{BASE}/outputs/predictions/error_cases.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
err[err["error_type"] == "correct"].sort_values("confidence").head(3)

In [ ]:
import shutil
backup = f"{DATASET}_labels_backup"
if not os.path.exists(backup):
    shutil.copytree(f"{DATASET}/labels", backup)

def poly_to_box(p):
    xy = list(map(float, p[1:]))
    xs, ys = xy[0::2], xy[1::2]
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return f"{p[0]} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}"

fixed = 0
for f in glob.glob(f"{DATASET}/labels/*/*.txt"):
    out, changed = [], False
    for line in open(f):
        p = line.split()
        if len(p) > 5:
            out.append(poly_to_box(p)); changed = True
        elif len(p) == 5:
            out.append(line.strip())
    if changed:
        open(f, "w").write("\n".join(out) + "\n"); fixed += 1
for c in glob.glob(f"{DATASET}/labels/*.cache"):
    os.remove(c)
print("label files fixed:", fixed)

In [ ]:
from IPython.display import Image, display
for f in ["confusion_matrix.png", "results.png"]:
    p = f"{RUN_DIR}/{f}"
    if os.path.exists(p):
        display(Image(p))

In [ ]:
os.makedirs(f"{BASE}/models", exist_ok=True)
shutil.copy2(best, f"{BASE}/models/best.pt")

onnx_path = YOLO(best).export(format="onnx", imgsz=640)
shutil.copy2(onnx_path, f"{BASE}/models/best.onnx")
print("Saved to", f"{BASE}/models")

In [ ]:
import os, glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO

BASE = "/content/drive/MyDrive/cv_project"
DATASET = f"{BASE}/dataset"

model = YOLO(f"{BASE}/models/best.pt")
preds = model.predict(
    source=f"{DATASET}/images/test",
    conf=0.35,
    save=True,
    project=f"{BASE}/outputs/predictions",
    name="test_predictions",
    exist_ok=True,
)
print("Saved to", preds[0].save_dir)

files = sorted(glob.glob(f"{preds[0].save_dir}/*.jpg"))[:15]
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for ax, f in zip(axes.ravel(), files):
    ax.imshow(mpimg.imread(f))
    ax.set_title(os.path.basename(f)[:20], fontsize=8)
    ax.axis("off")
for ax in axes.ravel()[len(files):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

rows = []
for r in preds:
    for b in r.boxes:
        rows.append({"image": os.path.basename(r.path),
                     "predicted": r.names[int(b.cls)],
                     "confidence": round(float(b.conf), 3)})
    if len(r.boxes) == 0:
        rows.append({"image": os.path.basename(r.path),
                     "predicted": "(nothing detected)",
                     "confidence": None})

df = pd.DataFrame(rows).sort_values("confidence", na_position="first")
df.to_csv(f"{BASE}/outputs/predictions/test_predictions.csv", index=False)
df.head(20)

In [ ]:
import numpy as np, os, pandas as pd

names = model.names

def load_gt(path, w, h):
    out = []
    if os.path.exists(path):
        for line in open(path):
            p = line.split()
            if len(p) < 5: continue
            c, xc, yc, bw, bh = int(p[0]), *map(float, p[1:5])
            out.append((c, [(xc - bw/2)*w, (yc - bh/2)*h, (xc + bw/2)*w, (yc + bh/2)*h]))
    return out

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union else 0

rows = []
for r in preds:
    img = os.path.basename(r.path)
    stem = os.path.splitext(img)[0]
    h, w = r.orig_shape
    gt = load_gt(f"{DATASET}/labels/test/{stem}.txt", w, h)
    pr = [(int(b.cls), float(b.conf), b.xyxy[0].tolist()) for b in r.boxes]
    used = set()
    for pc, conf, pb in sorted(pr, key=lambda x: -x[1]):
        best, bj = 0, None
        for j, (gc, gb) in enumerate(gt):
            if j in used: continue
            v = iou(pb, gb)
            if v > best: best, bj = v, j
        if best >= 0.5:
            used.add(bj)
            gc = gt[bj][0]
            rows.append([img, names[gc], names[pc], round(conf, 3), "correct" if gc == pc else "wrong class"])
        else:
            rows.append([img, "(nothing there)", names[pc], round(conf, 3), "false positive"])
    for j, (gc, gb) in enumerate(gt):
        if j not in used:
            rows.append([img, names[gc], "(missed)", None, "false negative"])

err = pd.DataFrame(rows, columns=["image", "actual", "predicted", "confidence", "error_type"])
print(err["error_type"].value_counts())
errors = err[err["error_type"] != "correct"]
errors.to_csv(f"{BASE}/outputs/predictions/error_analysis_raw.csv", index=False)
errors

In [ ]:
import os
from ultralytics import YOLO

BASE = "/content/drive/MyDrive/cv_project"
DATA_YAML = f"{BASE}/dataset/data.yaml"
RUN_DIR = f"{BASE}/outputs/training_results/yolov8n_baseline"
best = f"{RUN_DIR}/weights/best.pt"
model = YOLO(best)

for split in ["val", "test"]:
    m = model.val(data=DATA_YAML, split=split, imgsz=640, plots=True,
                  project=f"{BASE}/outputs/training_results", name=f"eval_{split}", exist_ok=True)
    print(f"\n=== {split.upper()} ===")
    print(f"Precision    : {m.box.mp:.3f}")
    print(f"Recall       : {m.box.mr:.3f}")
    print(f"mAP@0.5      : {m.box.map50:.3f}")
    print(f"mAP@0.5:0.95 : {m.box.map:.3f}")
    print("Per class (P, R, mAP50, mAP50-95):")
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        print(f"  {m.names[c]:10s} {p:.3f} {r:.3f} {ap50:.3f} {ap:.3f}")


In [ ]:
import os
from IPython.display import Image, display

RUN_DIR = "/content/drive/MyDrive/cv_project/outputs/training_results/yolov8n_baseline"
for f in ["confusion_matrix.png", "results.png"]:
    p = f"{RUN_DIR}/{f}"
    if os.path.exists(p):
        display(Image(p))